# Polar factorization as relabeling plus a Brenier map

This notebook generates `fig:monge-polar-factorization`.  It visualizes Brenier's polar factorization

$$
    u = \nabla \phi \circ s,
$$

where $s$ is measure preserving and $\nabla\phi$ is the convex-gradient transport factor.  We use a square grid, first apply a divergence-free swirling flow on the square, which preserves area up to the numerical time-integration accuracy and is tangent to the boundary, and then apply a symmetric positive definite affine stretch $x\mapsto Bx=\nabla(x^\top Bx/2)$.

The panels deliberately have no titles: the LaTeX source supplies the labels and caption.

In [1]:
from pathlib import Path
import shutil
import sys

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import to_rgb
from PIL import Image

ROOT = Path.cwd()
if not (ROOT / "notebooks-figures").exists():
    ROOT = ROOT.parent
sys.path.append(str(ROOT / "notebooks-figures"))

from figure_style import BLUE, RED, VIOLET, figure_dir, interp_color, remove_axes, save_pdf, setup_matplotlib

setup_matplotlib()

NAME = "monge-polar-factorization"
OUT = figure_dir(NAME)
ARXIV_OUT = ROOT / "arxiv" / "figures"
THUMB_OUT = ROOT / "notebooks-figures" / "thumbnails"
ARXIV_OUT.mkdir(parents=True, exist_ok=True)
THUMB_OUT.mkdir(parents=True, exist_ok=True)


## Two explicit factors

The relabeling is the time-one flow of a Hamiltonian vector field on the square.  With

$$
    \psi(x,y)=a\sin\left(\frac{\pi(x+1)}2\right)\sin\left(\frac{\pi(y+1)}2\right),
$$

we integrate

$$
    \dot x=\partial_y\psi(x,y),\qquad \dot y=-\partial_x\psi(x,y).
$$

This field is divergence free, so its exact flow preserves area.  It is also tangent to the boundary of the square, hence it acts as a relabeling of the source domain.  The Brenier factor is an affine map $x\mapsto Bx$ with $B=B^\top\succ0$, hence the gradient of the convex quadratic $x\mapsto x^\top Bx/2$.

In [2]:
twist_strength = 0.70
n_flow_steps = 96
stretch_angle = np.deg2rad(-22.0)
Q = np.array([[np.cos(stretch_angle), -np.sin(stretch_angle)], [np.sin(stretch_angle), np.cos(stretch_angle)]])
D = np.diag([1.40, 0.76])
B = Q @ D @ Q.T


def velocity(points):
    points = np.asarray(points)
    x = points[..., 0]
    y = points[..., 1]
    sx = np.sin(0.5 * np.pi * (x + 1.0))
    sy = np.sin(0.5 * np.pi * (y + 1.0))
    cx = np.cos(0.5 * np.pi * (x + 1.0))
    cy = np.cos(0.5 * np.pi * (y + 1.0))
    vx = twist_strength * 0.5 * np.pi * sx * cy
    vy = -twist_strength * 0.5 * np.pi * cx * sy
    return np.stack([vx, vy], axis=-1)


def relabel(points):
    """Area-preserving relabeling: RK4 integration of a divergence-free flow."""
    z = np.asarray(points, dtype=float).copy()
    dt = 1.0 / n_flow_steps
    for _ in range(n_flow_steps):
        k1 = velocity(z)
        k2 = velocity(z + 0.5 * dt * k1)
        k3 = velocity(z + 0.5 * dt * k2)
        k4 = velocity(z + dt * k3)
        z = z + (dt / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)
    return z


def brenier(points):
    points = np.asarray(points)
    return points @ B.T


def full_map(points):
    return brenier(relabel(points))


def jacobian_fd(fun, points, h=1e-5):
    points = np.asarray(points)
    e1 = np.array([h, 0.0])
    e2 = np.array([0.0, h])
    d1 = (fun(points + e1) - fun(points - e1)) / (2 * h)
    d2 = (fun(points + e2) - fun(points - e2)) / (2 * h)
    return d1[:, 0] * d2[:, 1] - d1[:, 1] * d2[:, 0]

rng = np.random.default_rng(43)
test = rng.uniform(-0.88, 0.88, size=(200, 2))
det_s = jacobian_fd(relabel, test)
print("finite-difference det(Ds):", float(det_s.min()), float(det_s.max()), float(det_s.mean()))
print("B eigenvalues:", np.linalg.eigvalsh(B))
assert np.allclose(det_s, 1.0, atol=1e-6)
assert np.all(np.linalg.eigvalsh(B) > 0)


finite-difference det(Ds): 0.9999999989719781 1.0000000012196648 1.0000000000871472
B eigenvalues: [0.76 1.4 ]


## Drawing the transformed grid

The grid carries a red-to-blue horizontal coloring so that the reader can see how labels are first scrambled by $s$ and then coherently stretched by the Brenier factor.  Faint arrows indicate the active displacement in each step.  The outer boundary is drawn slightly darker to make clear that the relabeling keeps the square domain fixed, while the Brenier factor turns it into an affine image.

In [3]:
def make_grid(n_lines=15, n_samples=420):
    values = np.linspace(-1, 1, n_lines)
    t = np.linspace(-1, 1, n_samples)
    lines = []
    for v in values:
        vertical = np.column_stack([np.full_like(t, v), t])
        horizontal = np.column_stack([t, np.full_like(t, v)])
        lines.append((vertical, v, "vertical"))
        lines.append((horizontal, v, "horizontal"))
    return lines


def line_color(v, kind):
    tau = (v + 1.0) / 2.0
    base = np.array(interp_color(tau, RED, BLUE))
    if kind == "horizontal":
        # Horizontal lines are paler so the vertical label colors remain readable.
        base = 0.64 * base + 0.36 * np.array(to_rgb("#ffffff"))
    return base


def transformed_lines(transform):
    return [(transform(line), v, kind) for line, v, kind in make_grid()]

panel_lines = {
    "source-grid": transformed_lines(lambda x: x),
    "relabeling-grid": transformed_lines(relabel),
    "final-grid": transformed_lines(full_map),
}

all_pts = np.vstack([line for lines in panel_lines.values() for line, _, _ in lines])
span = np.abs(all_pts).max() + 0.08
limits = (-span, span)
print("common plotting limits", limits)

arrow_base_1d = np.linspace(-0.72, 0.72, 6)
XX, YY = np.meshgrid(arrow_base_1d, arrow_base_1d, indexing="xy")
arrow_pts = np.column_stack([XX.ravel(), YY.ravel()])
rel_pts = relabel(arrow_pts)
fin_pts = brenier(rel_pts)


def draw_arrows(ax, tails, heads, *, alpha=0.24, scale=0.42):
    vec = heads - tails
    ax.quiver(
        tails[:, 0],
        tails[:, 1],
        scale * vec[:, 0],
        scale * vec[:, 1],
        angles="xy",
        scale_units="xy",
        scale=1,
        width=0.0025,
        headwidth=3.3,
        headlength=4.0,
        headaxislength=3.5,
        color="#000000",
        alpha=alpha,
        zorder=5,
    )


def draw_boundary(ax, transform):
    t = np.linspace(-1, 1, 520)
    boundary = [
        np.column_stack([np.full_like(t, -1), t]),
        np.column_stack([np.full_like(t, 1), t]),
        np.column_stack([t, np.full_like(t, -1)]),
        np.column_stack([t, np.full_like(t, 1)]),
    ]
    for line in boundary:
        z = transform(line)
        ax.plot(z[:, 0], z[:, 1], color="#1f2937", lw=0.78, alpha=0.78, zorder=4)


def draw_panel(key, transform, arrow_mode):
    fig, ax = plt.subplots(figsize=(2.45, 2.45))
    ax.set_aspect("equal")
    ax.set_xlim(limits)
    ax.set_ylim(limits)
    ax.set_facecolor("white")
    remove_axes(ax)

    for line, v, kind in panel_lines[key]:
        zorder = 2 if kind == "horizontal" else 3
        lw = 0.48 if kind == "horizontal" else 0.62
        ax.plot(line[:, 0], line[:, 1], color=line_color(v, kind), lw=lw, alpha=0.92, zorder=zorder)
    draw_boundary(ax, transform)

    if arrow_mode == "relabel":
        draw_arrows(ax, arrow_pts, rel_pts, alpha=0.52, scale=0.50)
    elif arrow_mode == "stretch":
        draw_arrows(ax, rel_pts, fin_pts, alpha=0.54, scale=0.37)
    elif arrow_mode == "arrive":
        # Short terminal arrows placed near the final locations keep the last panel readable.
        tails = fin_pts - 0.30 * (fin_pts - rel_pts)
        draw_arrows(ax, tails, fin_pts, alpha=0.48, scale=1.0)

    fig.tight_layout(pad=0.0)
    return fig

panels = [
    ("source-grid", lambda x: x, "relabel"),
    ("relabeling-grid", relabel, "stretch"),
    ("final-grid", full_map, "arrive"),
]

for key, transform, arrow_mode in panels:
    fig = draw_panel(key, transform, arrow_mode)
    pdf_path = OUT / f"{key}.pdf"
    save_pdf(fig, pdf_path, pad_inches=0.012)
    shutil.copyfile(pdf_path, ARXIV_OUT / f"{NAME}--{key}.pdf")
    plt.close(fig)

print("wrote", OUT)


common plotting limits (np.float64(-1.6124794146552472), np.float64(1.6124794146552472))


wrote /Users/gpeyre/Dropbox/github/ot4ml/latex/figures/monge-polar-factorization


## Thumbnail

The website thumbnail is a compact raster version of the same three panels, again without embedded text.

In [4]:
fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.3))
for ax, (key, transform, arrow_mode) in zip(axes, panels):
    ax.set_aspect("equal")
    ax.set_xlim(limits)
    ax.set_ylim(limits)
    ax.set_facecolor("white")
    remove_axes(ax)
    for line, v, kind in panel_lines[key]:
        zorder = 2 if kind == "horizontal" else 3
        lw = 0.45 if kind == "horizontal" else 0.57
        ax.plot(line[:, 0], line[:, 1], color=line_color(v, kind), lw=lw, alpha=0.92, zorder=zorder)
    draw_boundary(ax, transform)
    if arrow_mode == "relabel":
        draw_arrows(ax, arrow_pts, rel_pts, alpha=0.48, scale=0.50)
    elif arrow_mode == "stretch":
        draw_arrows(ax, rel_pts, fin_pts, alpha=0.50, scale=0.37)
    else:
        tails = fin_pts - 0.30 * (fin_pts - rel_pts)
        draw_arrows(ax, tails, fin_pts, alpha=0.42, scale=1.0)
fig.subplots_adjust(left=0.01, right=0.99, bottom=0.02, top=0.98, wspace=0.035)
thumb_path = THUMB_OUT / f"{NAME}.png"
fig.savefig(thumb_path, dpi=210, facecolor="white")
plt.close(fig)

# Keep thumbnails small enough for GitHub while preserving anti-aliased lines.
im = Image.open(thumb_path).convert("RGB")
im.thumbnail((980, 320), Image.Resampling.LANCZOS)
im.save(thumb_path, optimize=True)
print("thumbnail", thumb_path)


thumbnail /Users/gpeyre/Dropbox/github/ot4ml/notebooks-figures/thumbnails/monge-polar-factorization.png
